<a href="https://colab.research.google.com/github/muajnstu/Customer_Churn_Prediction/blob/main/Customer_Satisfaction_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/muajnstu/ML-Datasets/refs/heads/main/Customer_support_data.csv")
df.head()

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Unique id,channel_name,category,Sub-category,Customer Remarks,Order_id,order_date_time,Issue_reported at,issue_responded,Survey_response_Date,Customer_City,Product_category,Item_price,connected_handling_time,Agent_name,Supervisor,Manager,Tenure Bucket,Agent Shift,CSAT Score
0,7e9ae164-6a8b-4521-a2d4-58f7c9fff13f,Outcall,Product Queries,Life Insurance,NaN,c27c9bb4-fa36-4140-9f1f-21009254ffdb,NaN,01/08/2023 11:13,01/08/2023 11:47,01-Aug-23,NaN,NaN,NaN,NaN,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5
1,b07ec1b0-f376-43b6-86df-ec03da3b2e16,Outcall,Product Queries,Product Specific Information,NaN,d406b0c7-ce17-4654-b9de-f08d421254bd,NaN,01/08/2023 12:52,01/08/2023 12:54,01-Aug-23,NaN,NaN,NaN,NaN,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5
2,200814dd-27c7-4149-ba2b-bd3af3092880,Inbound,Order Related,Installation/demo,NaN,c273368d-b961-44cb-beaf-62d6fd6c00d5,NaN,01/08/2023 20:16,01/08/2023 20:38,01-Aug-23,NaN,NaN,NaN,NaN,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5
3,eb0d3e53-c1ca-42d3-8486-e42c8d622135,Inbound,Returns,Reverse Pickup Enquiry,NaN,5aed0059-55a4-4ec6-bb54-97942092020a,NaN,01/08/2023 20:56,01/08/2023 21:16,01-Aug-23,NaN,NaN,NaN,NaN,Patrick Flores,Olivia Wang,John Smith,>90,Evening,5
4,ba903143-1e54-406c-b969-46c52f92e5df,Inbound,Cancellation,Not Needed,NaN,e8bed5a9-6933-4aff-9dc6-ccefd7dcde59,NaN,01/08/2023 10:30,01/08/2023 10:32,01-Aug-23,NaN,NaN,NaN,NaN,Christopher Sanchez,Austin Johnson,Michael Lee,0-30,Morning,5


In [3]:
df.shape

(85907, 20)

In [ ]:
df.isna().sum()*100/df.shape[0]

In [ ]:
df.drop(columns=['Unique id','Order_id','order_date_time','Customer Remarks','Product_category','Item_price','connected_handling_time','Customer_City'],inplace=True)

In [ ]:
df.head()

### `Unique Values in Column`

In [ ]:
df.nunique()

In [ ]:
df.info()

In [ ]:
df['Issue_reported at'] = pd.to_datetime(df['Issue_reported at'], format='%d/%m/%Y %H:%M')
df['issue_responded'] = pd.to_datetime(df['issue_responded'], format='%d/%m/%Y %H:%M')
df['Survey_response_Date'] = pd.to_datetime(df['Survey_response_Date'], format='%d-%b-%y')

In [ ]:
print('Min Datetime        | Max Datetime')
print(df['Issue_reported at'].min(),'|',df['Issue_reported at'].max())
print(df['issue_responded'].min(),'|',df['issue_responded'].max())
print(df['Survey_response_Date'].min(),'|',df['Survey_response_Date'].max())

Looks like we have I month data

In [ ]:
df.info()

### `Feature Engineering`

In [ ]:
df.head()

In [ ]:
df['Issue_reported_at_hour'] = df['Issue_reported at'].dt.hour
df['issue_responded_hour'] = df['issue_responded'].dt.hour

In [ ]:
df.head()

In [ ]:
df_one_hot=pd.get_dummies(df['channel_name'],dtype=int)
df_one_hot

In [ ]:
# channel_name, Manager, Tenure Bucket, Agent Shift
df_channel = pd.get_dummies(df['channel_name'],dtype=int)
df_manager = pd.get_dummies(df['Manager'],dtype=int)
df_tenure = pd.get_dummies(df['Tenure Bucket'],dtype=int)
df_agent = pd.get_dummies(df['Agent Shift'],dtype=int)

In [ ]:
df = pd.concat([df,df_channel,df_manager,df_tenure,df_agent],axis=1)

In [ ]:
df.columns

In [ ]:
df.drop(columns=['channel_name','Manager', 'Tenure Bucket', 'Agent Shift','Issue_reported at','issue_responded','Survey_response_Date'],inplace=True)

In [ ]:
df.isna().sum()*100/df.shape[0]

In [ ]:
df.info()

## Target Encoding

In [ ]:
from category_encoders.target_encoder import TargetEncoder
columns=['category','Sub-category','Agent_name','Supervisor']
for i in columns:
    scaler = TargetEncoder()
    df[i]=scaler.fit_transform(X=df[i],y=df['CSAT Score'])
df.head()

## `Building a Model`

### 1. Linearity Check

In [ ]:
from scipy.stats import pearsonr
'''
Ho = There is no Linearity between Dependent and Independent Variables
Ha = There is Linearity between Dependent and Independent Variables
'''
alpha=0.05
for i in df.columns:
    pstat,pvalue=pearsonr(df[i],df['CSAT Score'])
    if pvalue<alpha:
        continue
    else:
        print(i,':Need to Drop')
        df.drop(columns=i,inplace=True)

This are the columns that don't have linearity with target field, so we drop it.

### 2. Multi-collinearity Check

In [ ]:
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df.drop(columns='CSAT Score',axis=1)
y = df['CSAT Score']
threshold = 10
default_vif = float('inf')
warnings.filterwarnings("ignore", message="divide by zero encountered in double_scalars")
while True:
    # Check if any VIF score is NaN (which will be caused by division by zero)
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

    # Handle division by zero
    vif_data.loc[vif_data['VIF'] == np.inf, 'VIF'] = default_vif

    high_vif_features = vif_data[vif_data["VIF"] > threshold]
    if high_vif_features.empty:
        break
    else:
        feature_to_drop = high_vif_features.iloc[0]['Feature']
        X = X.drop(columns=[feature_to_drop])
        print("Dropped column:", feature_to_drop)

In [ ]:
X

## `Train Test Split`

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [ ]:
scores = []

# Define the parameter grid and perform GridSearchCV
clf = GridSearchCV(
    DecisionTreeClassifier(),
    {
        'criterion': ['entropy', 'gini']
    },
    cv=5
)

clf.fit(X_train, y_train)

# Append the results to the scores list
scores.append({'model': 'DecisionTree', 'best_score': clf.best_score_ * 100, 'best_param': clf.best_params_})

# Create a DataFrame from the scores list and print it
result_df = pd.DataFrame(scores, columns=['model', 'best_score', 'best_param'])
print(result_df)
model = DecisionTreeClassifier(criterion='entropy')
model.fit(X_train,y_train)
plt.figure(figsize=(15,10))
plot_tree(model, filled=True, feature_names=X_train.columns)
plt.show()

In [ ]:
model = KNeighborsClassifier(n_neighbors=27)
model.fit(X_train,y_train)

N neighbor 27 was determined using GridSearchCV

In [ ]:
y_test_pred=model.predict(X_test)
print('Accuracy:',round(accuracy_score(y_test,y_test_pred)*100,4),'%')

In [ ]:
model = LogisticRegression(penalty='l2',C=0.01,multi_class='ovr')
model.fit(X_train,y_train)

In [ ]:
y_test_pred=model.predict(X_test)
print('Accuracy:',round(accuracy_score(y_test,y_test_pred)*100,4),'%')